# Citation highlighting — highlights that land on the words, not the chunk

Most RAG citation UIs can only point at the *chunk* that was retrieved — a
500-token block the user has to re-read. [TokenPath](https://tokenpath.ai)
returns character offsets on **both** sides: which characters of the answer, and
which characters of the source, at token-level granularity. Your highlights land
on the exact words that carried the claim.

This notebook renders a cited answer two ways:

1. inline in the notebook, and
2. as a standalone `citations.html` with click-to-scroll — view-source-friendly
   starter markup for your own UI.

It also shows the **bracket-and-link** pattern: instead of making the model emit
`[1]`-style citations while it generates, ask it only to wrap citable spans in
`[[…]]` — a much easier task — and let attribution do the actual linking in
post-processing.

## Setup

You need a TokenPath API key — free at [platform.tokenpath.ai](https://platform.tokenpath.ai)
(10M attributed tokens on signup, no card required). Export it as `TOKENPATH_API_KEY`
before running this notebook.

In [1]:
%pip install -q requests

In [2]:
import os
import re

import requests

API_URL = "https://api.tokenpath.ai"
API_KEY = os.environ.get("TOKENPATH_API_KEY")
assert API_KEY, (
    "Set TOKENPATH_API_KEY first — grab a free key (10M tokens, no card) "
    "at https://platform.tokenpath.ai"
)


def attribute(document, question, answer, spans, **options):
    """Resolve each answer character span to the source span that produced it."""
    response = requests.post(
        f"{API_URL}/v1/attributions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "document": document,
            "question": question,
            "answer": answer,
            "spans": spans,
            **options,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

## Sample document and answer

In [3]:
DOCUMENT = """NORTHWIND TRADERS — Q3 2025 SHAREHOLDER NOTE

Revenue grew 18% year over year to $412 million, driven primarily by the
Enterprise segment, which expanded 31%. Gross margin improved to 64.2%,
up from 61.8% a year ago. The company closed the quarter with 2,847
employees, up from 2,610 at the end of Q2.

Operating expenses rose 9% to $198 million, reflecting continued
investment in the Fabrikam integration, which remains on track to
complete in Q1 2026. The board approved a $50 million share buyback
program. No dividend was declared this quarter.

CEO Elena Vasquez said the company expects "mid-teens revenue growth"
for the full fiscal year, citing a record $1.2 billion contracted
backlog. Guidance assumes no material FX headwinds."""

QUESTION = "Summarize Northwind's Q3 results."

In [4]:
ANSWER = (
    "Northwind's revenue grew 18% year over year to $412 million, led by 31% "
    "growth in the Enterprise segment. Gross margin improved to 64.2%. The "
    "board approved a $50 million share buyback, and the company guided to "
    "mid-teens revenue growth for the full year."
)


def claim_spans(text):
    """Sentence-level [start, end) character spans over `text`."""
    boundary = re.compile(r"[.!?][\"\')\]]*(?=\s|$)|\n")
    raw, start = [], 0
    for m in boundary.finditer(text):
        raw.append((start, m.end()))
        start = m.end()
    if start < len(text):
        raw.append((start, len(text)))
    spans = []
    for s, e in raw:
        segment = text[s:e]
        if segment.strip():
            left = len(segment) - len(segment.lstrip())
            right = len(segment) - len(segment.rstrip())
            spans.append([s + left, e - right])
    return spans


resolved = attribute(DOCUMENT, QUESTION, ANSWER, claim_spans(ANSWER))
for item in resolved["spans"]:
    source = item["source"]
    print(f"[{source['confidence']:.2f}] {item['answer']['text'][:48]!r}")
    print(f"       -> chars {source['start']}–{source['end']}: {source['text']!r}")

[0.99] "Northwind's revenue grew 18% year over year to $"
       -> chars 54–157: 'grew 18% year over year to $412 million, driven primarily by the\nEnterprise segment, which expanded 31%'
[0.98] 'Gross margin improved to 64.2%.'
       -> chars 159–189: 'Gross margin improved to 64.2%'
[0.99] 'The board approved a $50 million share buyback, '
       -> chars 595–646: 'mid-teens revenue growth"\nfor the full fiscal year,'


## Render it inline

Each claim gets a numbered tint; the same number and tint mark the exact source
characters it resolved to. Note the highlights cover *words*, not chunks.

In [5]:
import html as html_lib

from IPython.display import HTML, display

TINTS = ["#e6dcf7", "#d5e8d4", "#fbe5c8", "#f6d6e0"]


def render_citations(document, answer, resolved):
    items = [item for item in resolved["spans"] if item["source"]]

    def paint(text, spans, tag):
        parts, cursor = [], 0
        for n, start, end, tint, confidence in spans:
            parts.append(html_lib.escape(text[cursor:start]))
            parts.append(
                f'<{tag} style="background:{tint};color:#1a1a1a;border-radius:3px;'
                f'padding:0 2px" title="confidence {confidence:.2f}">'
                f"{html_lib.escape(text[start:end])}</{tag}><sup>{n}</sup>"
            )
            cursor = end
        parts.append(html_lib.escape(text[cursor:]))
        return "".join(parts)

    answer_spans, doc_spans = [], []
    for i, item in enumerate(items):
        tint = TINTS[i % len(TINTS)]
        confidence = item["source"]["confidence"]
        answer_spans.append(
            (i + 1, item["answer"]["start"], item["answer"]["end"], tint, confidence)
        )
        doc_spans.append(
            (i + 1, item["source"]["start"], item["source"]["end"], tint, confidence)
        )
    doc_spans.sort(key=lambda s: s[1])
    doc_spans = [
        s for j, s in enumerate(doc_spans) if j == 0 or s[1] >= doc_spans[j - 1][2]
    ]  # this simple renderer skips overlapping source spans

    return (
        '<div style="font-family:Menlo,Consolas,monospace;font-size:13px;'
        'line-height:1.8;max-width:720px">'
        f"<p><b>Answer</b></p><p>{paint(answer, answer_spans, 'span')}</p>"
        f"<p><b>Source</b></p>"
        f'<p style="white-space:pre-wrap">{paint(document, doc_spans, "mark")}</p>'
        "</div>"
    )


display(HTML(render_citations(DOCUMENT, ANSWER, resolved)))

## Save a standalone page with click-to-scroll

The same data, as a self-contained HTML file: click a citation number in the
answer and the page scrolls to (and flashes) the exact source span.

In [6]:
def citation_page(document, answer, resolved):
    items = [item for item in resolved["spans"] if item["source"]]

    parts, cursor = [], 0
    for i, item in enumerate(items):
        a = item["answer"]
        parts.append(html_lib.escape(answer[cursor : a["start"]]))
        parts.append(
            f'{html_lib.escape(answer[a["start"] : a["end"]])}'
            f'<a href="#src-{i}" class="cite">[{i + 1}]</a>'
        )
        cursor = a["end"]
    parts.append(html_lib.escape(answer[cursor:]))
    answer_html = "".join(parts)

    doc_items = sorted(enumerate(items), key=lambda p: p[1]["source"]["start"])
    parts, cursor = [], 0
    for i, item in doc_items:
        s = item["source"]
        if s["start"] < cursor:
            continue
        parts.append(html_lib.escape(document[cursor : s["start"]]))
        parts.append(
            f'<mark id="src-{i}">{html_lib.escape(document[s["start"] : s["end"]])}</mark>'
        )
        cursor = s["end"]
    parts.append(html_lib.escape(document[cursor:]))
    doc_html = "".join(parts)

    return f"""<!doctype html><meta charset="utf-8"><title>TokenPath citations</title>
<style>
  body {{ font: 14px/1.8 Menlo, Consolas, monospace; max-width: 720px;
         margin: 40px auto; padding: 0 16px; color: #1a1a1a; }}
  .cite {{ text-decoration: none; color: #5b4a8a; font-weight: bold; }}
  mark {{ background: #e6dcf7; border-radius: 3px; padding: 0 2px;
          scroll-margin-top: 40px; }}
  mark:target {{ background: #c9b6ee; }}
  pre {{ white-space: pre-wrap; }}
</style>
<h3>Answer</h3><p>{answer_html}</p>
<h3>Source</h3><pre>{doc_html}</pre>"""


with open("citations.html", "w") as f:
    f.write(citation_page(DOCUMENT, ANSWER, resolved))
print("wrote citations.html — open it in a browser")

wrote citations.html — open it in a browser


## Bracket-and-link: let the model mark, let attribution link

Perplexity-style `[1][2]` citations make the model do extra work while
generating — decide what to cite, track source ids, emit the right marker.
Flip it: prompt the model to just **wrap spans that need a citation in
`[[…]]`**, then attribute exactly those spans and wire up the links yourself.
The model's job shrinks to "mark what needs citing"; TokenPath answers "where
it points".

In [7]:
# e.g. generated with: "...wrap every factual claim that needs a citation in [[ ]]"
BRACKETED = (
    "Northwind grew revenue [[18% year over year]], with the Enterprise segment "
    "[[expanding 31%]], and the board approved a [[$50 million share buyback]]."
)


def parse_brackets(text):
    """Strip [[...]] markers; return (clean_text, spans) in clean-text coordinates."""
    clean, spans, cursor = "", [], 0
    for m in re.finditer(r"\[\[(.+?)\]\]", text):
        clean += text[cursor : m.start()]
        spans.append([len(clean), len(clean) + len(m.group(1))])
        clean += m.group(1)
        cursor = m.end()
    return clean + text[cursor:], spans


clean_answer, spans = parse_brackets(BRACKETED)
linked = attribute(DOCUMENT, QUESTION, clean_answer, spans)
for item in linked["spans"]:
    source = item["source"]
    print(
        f"{item['answer']['text']!r:38} -> {source['text']!r}"
        f"  (confidence {source['confidence']:.2f})"
    )

'18% year over year'                   -> '18% year over year'  (confidence 0.98)
'expanding 31%'                        -> 'expanded 31%'  (confidence 0.84)
'$50 million share buyback'            -> '$50 million share buyback'  (confidence 0.99)


## Production notes

- **Offsets are exact.** `answer.start/end` and `source.start/end` are character
  offsets into the strings you sent — safe to slice with directly, no fuzzy
  matching needed.
- **Confidence per citation.** Display it alongside each citation, or compare it
  across links while debugging attribution. It describes relative attribution
  strength; every resolved citation remains available.
- **Multiple documents.** Attribute the answer against each source document (or
  concatenate with clear separators and map offsets back). The
  [PDF example](https://docs.tokenpath.ai) extends this to page + bounding-box
  highlights.

---

## Where to go next

- **API reference & guides** — [docs.tokenpath.ai](https://docs.tokenpath.ai)
- **Bugs / broken examples** — [open an issue](https://github.com/tokenpath/tokenpath-cookbook/issues)
- **"How do I…?"** — [start a discussion](https://github.com/tokenpath/tokenpath-cookbook/discussions)
- **Email** — support@tokenpath.ai